# AI Sommelier — тестирование диалогов

Ручной стенд для turn-based агента. Notebook использует настроенную модель из `llm_module.py`, но не читает и не выводит API-ключ.

Каждый сценарий можно запускать с чистой памятью. После диалога показываются сохранённые `TurnMemory` и `UserProfile`.

In [1]:
from pathlib import Path
import os
import sys
from pprint import pprint

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "sommelier").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from sommelier.agent.graph import run_agent_turn
from sommelier.agent.state import AgentState
from sommelier.storage.session_repository import get_default_repository

repository = get_default_repository()
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\egorg\Documents\SPIRT_TEST


In [2]:
SESSION_ID = "notebook-dialog-testing-v4"

def reset_session(session_id: str = SESSION_ID) -> None:
    repository.delete_session(session_id)
    print(f"Cleared session: {session_id}")

def ask(message: str, session_id: str = SESSION_ID):
    state = run_agent_turn(
        AgentState(session_id=session_id, user_request=message),
        config={"configurable": {"repository": repository}},
    )
    resolution = state.turn_resolution
    answer = state.final_answer_result

    print("\n" + "=" * 88)
    print(f"USER: {message}")
    print(f"FOLLOW UP: {resolution.follow_up if resolution else None}")
    print(f"INITIAL REQUEST: {resolution.initial_request if resolution else None}")
    print(f"EFFECTIVE REQUEST: {resolution.effective_request if resolution else None}")
    print(f"NEGATIVE REQUEST: {resolution.negative_request if resolution else None}")
    print(f"TOOL CALLS: {state.tool_call_count}")
    print("TOOL TRACE:")
    for trace in state.tool_traces:
        if trace.tool_name == "tool_call":
            print(f"  {trace.input.get('tool')}: {trace.status} — {trace.output_summary}")
    print(f"FULL CARDS THIS TURN: {[(card.kind, card.id) for card in state.cards]}")
    print(f"USED REFS: {[ref.model_dump() for ref in answer.used_result_refs] if answer else []}")
    print(f"ASSISTANT: {answer.answer if answer else None}")
    print(f"ERRORS: {state.errors}")
    return state

def run_dialog(messages: list[str], session_id: str = SESSION_ID, reset: bool = True):
    if reset:
        reset_session(session_id)
    states = [ask(message, session_id) for message in messages]
    assert all(state.tool_call_count <= 2 for state in states)
    return states

def show_saved_state(session_id: str = SESSION_ID) -> None:
    memory = repository.load_session_memory(session_id)
    profile = repository.load_user_profile(session_id)
    print("\nSAVED TURN MEMORY:")
    pprint(memory.model_dump(mode="json"))
    print("\nSAVED USER PROFILE:")
    pprint(profile.model_dump(mode="json"))
    print("\nFULL CHAT TRANSCRIPT:")
    pprint(repository.load_messages(session_id))
    print("\nSAVED TRACES:")
    pprint(repository.load_trace_events(session_id))

## Сценарий 4 — корзина

Проверяет добавление ранее показанного продукта, увеличение количества, просмотр и полное удаление позиции.

In [3]:
CART_SESSION_ID = "notebook-cart-testing-v4"
cart_states = run_dialog(
    [
        "Посоветуй выдержанный ром для Old Fashioned.",
        "Добавь первый из последних вариантов в корзину, 2 штуки.",
        "Добавь туда же ещё одну штуку.",
        "Покажи, что сейчас в корзине.",
        "Убери этот ром из корзины.",
        "Покажи корзину ещё раз.",
    ],
    session_id=CART_SESSION_ID,
    reset=True,
)
show_saved_state(CART_SESSION_ID)

Cleared session: notebook-cart-testing-v4

USER: Посоветуй выдержанный ром для Old Fashioned.
FOLLOW UP: False
INITIAL REQUEST: Посоветуй выдержанный ром для Old Fashioned.
EFFECTIVE REQUEST: Подобрать выдержанный ром для коктейля Old Fashioned.
NEGATIVE REQUEST: None
TOOL CALLS: 1
TOOL TRACE:
  search_products: success — returned_after_filter=7; cards_in_current_turn=7
FULL CARDS THIS TURN: [('product', 'bacardi-spiced-rum'), ('product', 'bacardi-anejo-cuatro-rum'), ('product', 'bacardi-reserva-ocho-rum'), ('product', 'bacardi-gran-reserva-limitada'), ('product', 'bacardi-carta-oro'), ('product', 'bacardi-gran-reserva-diez'), ('product', 'bacardi-carta-negra')]
USED REFS: [{'kind': 'product', 'id': 'bacardi-reserva-ocho-rum'}]
ASSISTANT: Конечно, для Old Fashioned я бы взял BACARDÍ Reserva Ocho Rum. У него как раз тот профиль, который хорошо держит коктейль: зрелые ноты сушёных фруктов, специй и дубово-ванильная глубина. Он уже сам по себе округлый и многослойный, поэтому в Old Fashio

## Сценарий 1 — поиск, follow-up и lookup по порядку

Ожидание: третий запрос должен определить первый объект в `shown_results` предыдущего хода и вызвать `lookup_by_id` для его полной карточки.

In [ ]:
cocktail_dialog = [
    "Посоветуй свежий коктейль с лаймом и мятой.",
    "А есть что-то ещё, но не сладкое?",
    "Как приготовить первый из последних вариантов?",
]

cocktail_states = run_dialog(cocktail_dialog)
show_saved_state()

## Сценарий 2 — новый запрос после коктейльной темы

Ожидание: запрос про ром имеет `follow_up=False`, даже если запускается в той же сессии.

In [ ]:
product_state = ask(
    "Теперь посоветуй выдержанный ром для коктейлей, но не сладкий.",
    SESSION_ID,
)
show_saved_state()

## Сценарий 3 — профиль без поиска

Ожидание: preference сохраняется, search tools не вызываются.

In [ ]:
PROFILE_SESSION_ID = "notebook-profile-testing-v4"
profile_states = run_dialog(
    [
        "Запомни, что я люблю дубовые и пряные вкусы.",
        "А теперь посоветуй ром для Old Fashioned.",
    ],
    session_id=PROFILE_SESSION_ID,
)
show_saved_state(PROFILE_SESSION_ID)

## Свой диалог

Измените список и запустите ячейку. `reset=True` начинает с пустой memory/profile.

In [10]:
custom_dialog = [
    "Посоветуй ром к стейку.",
    "А есть вариант без выраженной сладости?",
]

# custom_states = run_dialog(custom_dialog, session_id="my-custom-dialog", reset=True)
# show_saved_state("my-custom-dialog")